# 01. 학습 최적화 템플릿 (RTX 4070 SUPER 12GB)
TF32 · BF16 autocast · 고속 DataLoader · CUDA 스트림 프리페치 · fused AdamW · Gradient accumulation · Gradient checkpointing · torch.compile 을 **켜고 끄며 처리량을 비교**합니다.
- 합성 데이터라 다운로드 없이 바로 실행됩니다. 실제 과제에서는 `SyntheticImages` 만 내 Dataset 으로 바꾸세요.
- 커널 시작 시 `00-perf.py` 가 TF32·cudnn.benchmark·메모리 얼로케이터를 자동 설정합니다.

In [ ]:
import os, time, torch, torch.nn as nn, torchvision
from torch.utils.data import Dataset, DataLoader
print(torch.__version__, torch.cuda.get_device_name(0))
print("TF32 matmul:", torch.backends.cuda.matmul.allow_tf32, "| cudnn.benchmark:", torch.backends.cudnn.benchmark)
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
DEV = "cuda"

In [ ]:
class SyntheticImages(Dataset):
    """CPU 에서 이미지를 만드는 가짜 데이터셋 (실제 디코딩 부하 흉내)"""
    def __init__(self, n=4096, size=224, classes=1000):
        self.n, self.size, self.classes = n, size, classes
    def __len__(self): return self.n
    def __getitem__(self, i):
        g = torch.Generator().manual_seed(i)
        return torch.rand(3, self.size, self.size, generator=g), i % self.classes

def make_loader(batch, fast=True):
    ds = SyntheticImages()
    if not fast:
        return DataLoader(ds, batch_size=batch, shuffle=True, num_workers=0)
    return DataLoader(ds, batch_size=batch, shuffle=True,
                      num_workers=min(8, os.cpu_count() // 2),  # CPU 전처리를 병렬로
                      pin_memory=True,                           # 고정 메모리 → 비동기 H2D 복사
                      persistent_workers=True, prefetch_factor=4, drop_last=True)

class CUDAPrefetcher:
    """별도 CUDA 스트림에서 다음 배치를 미리 GPU 로 올려 계산과 복사를 겹친다"""
    def __init__(self, loader):
        self.loader, self.stream = loader, torch.cuda.Stream()
    def __iter__(self):
        it, nxt = iter(self.loader), None
        def load():
            try: x, y = next(it)
            except StopIteration: return None
            with torch.cuda.stream(self.stream):
                return (x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last),
                        y.to(DEV, non_blocking=True))
        nxt = load()
        while nxt is not None:
            torch.cuda.current_stream().wait_stream(self.stream)
            cur, nxt = nxt, load()
            yield cur

In [ ]:
from torch.utils.checkpoint import checkpoint
class Ckpt(nn.Module):
    """블록 활성값을 저장하지 않고 역전파 때 재계산 → VRAM 절약, 더 큰 배치"""
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return checkpoint(self.m, x, use_reentrant=False)

def build(ckpt=False, compile_=False):
    m = torchvision.models.resnet50()
    if ckpt: m.layer3, m.layer4 = Ckpt(m.layer3), Ckpt(m.layer4)
    m = m.to(DEV, memory_format=torch.channels_last)
    return torch.compile(m) if compile_ else m

def run(name, batch=64, steps=30, fast_loader=True, prefetch=True, amp=True,
        fused=True, accum=1, ckpt=False, compile_=False):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = build(ckpt, compile_)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, fused=fused)
    loss_fn = nn.CrossEntropyLoss()
    loader = make_loader(batch, fast_loader)
    data = CUDAPrefetcher(loader) if prefetch else ((x.to(DEV), y.to(DEV)) for x, y in loader)
    seen, t0 = 0, None
    for i, (x, y) in enumerate(data):
        if i == 5: torch.cuda.synchronize(); t0 = time.time(); seen = 0   # 워밍업(컴파일 포함) 제외
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
            loss = loss_fn(model(x), y) / accum
        loss.backward()
        if (i + 1) % accum == 0:
            opt.step(); opt.zero_grad(set_to_none=True)
        seen += x.size(0)
        if i >= steps + 5: break
    torch.cuda.synchronize()
    ips = seen / (time.time() - t0)
    mem = torch.cuda.max_memory_allocated() / 2**30
    print(f"{name:34s} {ips:8.1f} img/s   최대 VRAM {mem:5.2f} GB")
    return ips

In [ ]:
# 하나씩 켜 가며 비교 (각 30스텝, 수십 초~수 분)
res = {}
res["기본 (FP32, 단일 로더)"]        = run("기본 (FP32, 단일 로더)", fast_loader=False, prefetch=False, amp=False, fused=False)
res["+ 고속 DataLoader·프리페치"]    = run("+ 고속 DataLoader·프리페치", amp=False, fused=False)
res["+ BF16 autocast + fused AdamW"] = run("+ BF16 autocast + fused AdamW")
res["+ torch.compile"]               = run("+ torch.compile", compile_=True)
res["+ checkpointing (배치 2배)"]    = run("+ checkpointing (배치 2배)", batch=128, ckpt=True)
base = list(res.values())[0]
for k, v in res.items(): print(f"{k:34s} {v/base:5.2f}x")

### Gradient accumulation — 12GB 로 큰 실효 배치
`run(..., batch=32, accum=8)` → 실효 배치 256. 메모리는 배치 32 만큼만 씁니다.

### FP16 을 써야 할 때는 GradScaler 필수 (BF16 은 불필요)
```python
scaler = torch.amp.GradScaler("cuda")
with torch.autocast("cuda", dtype=torch.float16):
    loss = loss_fn(model(x), y)
scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
```

In [ ]:
# 병목 찾기 1: torch.profiler 요약 (노트북 안에서 바로)
from torch.profiler import profile, ProfilerActivity
model = build(); x = torch.rand(64, 3, 224, 224, device=DEV).to(memory_format=torch.channels_last)
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    with torch.autocast("cuda", dtype=torch.bfloat16):
        model(x).sum().backward()
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=12))

### 병목 찾기 2: Nsight Systems (타임라인으로 확인)
노트북 코드를 `train.py` 로 저장한 뒤 터미널에서:
```bash
nsys profile -o /tmp/train --trace=cuda,nvtx,osrt python train.py
```
구간 표시는 `torch.cuda.nvtx.range_push("forward")` / `range_pop()`.
커널 단위 상세 분석(`ncu`)은 호스트에 프로파일링 허용 옵션이 반영된 뒤(다음 콜드 부팅) 가능합니다.